# Decode throughput — is the O(1)-state inference advantage real?

`CubbyModel.step()` carries a fixed-size state (MinGRU) or a windowed KV cache
(hybrid) — bounded either way, unlike a growing KV cache. This measures it.

- **Arm A** (always): incremental `step()` vs naive full-prefix recompute, same
  model. The `step()` ms/tok should stay ~flat as context grows; naive climbs.
- **Arm B** (optional): vs a real peer (Qwen2.5-1.5B). Expect to lose at short
  context (Python loop vs fused kernels); the *crossover* is the finding.

In [ ]:
# --- setup: clone repo, deps, mount Drive (run once per session) ---
import os, subprocess, sys, time
if not os.path.exists('/content/CubbyLLM'):
    !git clone -q https://github.com/Grillcheese-AI/CubbyLLM.git /content/CubbyLLM
else:
    !cd /content/CubbyLLM && git pull -q --ff-only
!pip -q install torch numpy sentencepiece
from google.colab import drive; drive.mount('/content/drive')
REPO = '/content/CubbyLLM'
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# --- EDIT to your Drive locations ---
DRIVE     = '/content/drive/MyDrive/cubbyllm'
TOKENIZER = f'{DRIVE}/grillcheese_bbpe128k.json'   # your tokenizer .json/.model
CORPUS_DRIVE = f'{DRIVE}/token_cache'              # uint32 shards on Drive
CORPUS    = '/content/token_cache'                 # staged to LOCAL SSD

# Stage the corpus to local SSD. Drive-FUSE memmap reads bottleneck the GPU —
# this run measured 3,000 tok/s off Drive vs ~100,000 tok/s off local disk.
# Skip (comment out) if CORPUS is already populated this session.
!mkdir -p {CORPUS} && rsync -a --info=progress2 {CORPUS_DRIVE}/ {CORPUS}/
!du -sh {CORPUS}

In [ ]:
# --- run a script, tee to a log, catch a hung child on interrupt ---
def run(script, env_extra, log_name):
    env = dict(os.environ, CUBBY_SPM=TOKENIZER, CB_CORPUS=CORPUS, **env_extra)
    os.makedirs(f'{REPO}/validation/logs', exist_ok=True)
    log = f'{REPO}/validation/logs/{log_name}'
    p = None
    try:
        with open(log, 'a', encoding='utf-8', buffering=1) as f:
            f.write(f"\n=== {time.strftime('%F %T')} "
                    + ' '.join(f'{k}={v}' for k, v in sorted(env_extra.items())) + '\n')
            p = subprocess.Popen([sys.executable, '-u', f'validation/{script}'],
                                 cwd=REPO, env=env, stdout=subprocess.PIPE,
                                 stderr=subprocess.STDOUT, text=True, bufsize=1)
            for line in p.stdout:
                print(line, end=''); f.write(line)
            p.wait()
    finally:
        if p and p.poll() is None:      # never leave a child holding the GPU
            p.terminate()

In [ ]:
# --- Arm A only (no download). Point CB_CKPT at a trained checkpoint. ---
run('exp_decode_throughput.py',
    dict(CB_CKPT=f'{DRIVE}/ab_hybrid.pt', CB_CTXS='512,2048,8192,32768'),
    'decode_hybrid.log')

In [ ]:
# --- Arm B: vs Qwen2.5-1.5B (downloads ~3GB, needs transformers) ---
!pip -q install transformers
run('exp_decode_throughput.py',
    dict(CB_CKPT=f'{DRIVE}/ab_hybrid.pt', CB_CTXS='512,2048,8192,32768',
         CB_PEER='Qwen/Qwen2.5-1.5B'),
    'decode_vs_peer.log')

### How to read
Arm A: a flat `step()` ms/tok across contexts **is** the O(1) claim, measured;
the speedup over naive grows with context. Arm B: losing at short context is
expected — report the crossover length, not the headline.